# Homework 6: Humanoid Motion Imitation

In this assignment you will train a **Unitree G1 humanoid robot** to imitate a short reference motion clip (~2–3 seconds) using **PPO** in GPU-accelerated simulation (`mujoco-warp`), then **deploy** the trained policy in a separate vanilla MuJoCo simulator to verify sim-to-sim transfer.

## Learning Objectives

1. **Observation noise & deployable observations** — understand why certain observations (e.g. base linear velocity) are unavailable on real hardware and how sensor noise affects policy robustness.
2. **Domain randomization** — implement and ablate randomization of physical parameters (friction, COM, encoder bias, external pushes) that bridge the gap between training and deployment simulators.
3. **Terminal conditions** — design early-termination criteria that shape the learning landscape and prevent the policy from entering irrecoverable states.

## Pipeline Overview

```
Motion CSV → NPZ (preprocessing)
    ↓
Train policy (mujoco-warp, GPU, 4096 envs)
    ↓
Deploy policy (vanilla mujoco, CPU, 16 runs) ← sim-to-sim
    ↓
Evaluate tracking error & render video
```

## Compute Requirements

- **Runtime**: Google Colab **T4 GPU** (free tier). Select *Runtime → Change runtime type → T4 GPU*.
- **Training time**: up to ~2 hours per training run (1500 iterations, 4096 environments) on a T4. The baseline plus the three Part 4 ablations need about 5 hours of GPU time in total. Start early.

---
## Part 0: Environment Setup

In [ ]:
#@title 0.0 Mounting to Drive { display-mode: "form" }

import os
try:  # Google Colab
    from google.colab import drive
    drive.mount('/content/drive/')
    folder_path = '/content/drive/My Drive/10703_HW6'
except ImportError:  # Babel / local: run from the project folder
    folder_path = os.environ.get('HW6_DIR', os.getcwd())
print(os.listdir(folder_path))
os.chdir(folder_path)
print('cwd:', os.getcwd())

In [ ]:
#@title 0.1 Install Dependencies { display-mode: "form" }

import subprocess, os, sys

# --- Check GPU availability ---
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu_info.returncode != 0:
    raise RuntimeError("No GPU detected. Please enable a GPU runtime: "
                       "Runtime > Change runtime type > T4 GPU")
print(f"GPU: {gpu_info.stdout.strip()}")

# Headless rendering
os.environ["MUJOCO_GL"] = "egl"

!pip install -q -e .

from importlib.metadata import version as pkg_version
print(f"Python {sys.version.split()[0]}")
for pkg in ["mjlab", "rsl-rl-lib", "mujoco", "mujoco-warp", "warp-lang", "torch"]:
    print(f"  {pkg:<12} {pkg_version(pkg)}")

print("\n--- Setup complete ---")

In [ ]:
#@title 0.2 Imports { display-mode: "form" }

import copy, glob, base64, math
from pathlib import Path
from datetime import datetime
from dataclasses import asdict

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from mjlab.envs import ManagerBasedRlEnv, ManagerBasedRlEnvCfg
from mjlab.rl import (
    RslRlOnPolicyRunnerCfg,
    RslRlModelCfg,
    RslRlPpoAlgorithmCfg,
    MjlabOnPolicyRunner,
    RslRlVecEnvWrapper,
)
from src.tasks.tracking.config.g1.env_cfgs import unitree_g1_flat_tracking_env_cfg
from src.tasks.tracking.config.g1.rl_cfg import unitree_g1_tracking_ppo_runner_cfg
from src.tasks.tracking.mdp import MotionCommand, MotionCommandCfg
from mjlab.utils.torch import configure_torch_backends

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SEED = 42

configure_torch_backends()
print(f"Device: {DEVICE}  |  PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")


# Create video
def show_mp4(path: str, width: int = 640):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f'</video>'
    ))

In [ ]:
#@title 0.3 Prepare a Short Motion Clip (2–3 seconds) { display-mode: "form" }

# The full dance clip is ~131 s at 30 fps (3944 frames). We extract a
# short 2.5-second segment (CSV rows 400–475 at 30 fps) so that training
# converges quickly on a free-tier GPU.

MOTION_CSV = Path("src/assets/motions/g1/dance1_subject2.csv")
CLIP_NAME = "dance1_short"
MOTION_NPZ = Path(f"src/assets/motions/g1/{CLIP_NAME}.npz")

# CSV line range: rows 400–475 (75 frames at 30 fps = 2.5 seconds)
LINE_START, LINE_END = 400, 475
INPUT_FPS, OUTPUT_FPS = 30, 50

print("Converting CSV motion segment to NPZ...")
!python scripts/csv_to_npz.py \
    --input-file {MOTION_CSV} \
    --output-name {CLIP_NAME} \
    --input-fps {INPUT_FPS} \
    --output-fps {OUTPUT_FPS} \
    --line-range '({LINE_START},{LINE_END})' \
    --device {DEVICE}

data = np.load(str(MOTION_NPZ))
fps = int(data["fps"][0])
n_frames = data["joint_pos"].shape[0]
duration = n_frames / fps

print(f"\n--- Motion Clip Statistics ---")
print(f"  FPS:        {fps}")
print(f"  Frames:     {n_frames}")
print(f"  Duration:   {duration:.2f}s")
print(f"  Joints:     {data['joint_pos'].shape[1]}")
print(f"  Bodies:     {data['body_pos_w'].shape[1]}")

In [ ]:
#@title 0.4 Visualize Reference Motion { display-mode: "form" }

# Replay the processed kinematic motion through MuJoCo and render a video
# so we can see what the policy will be trained to imitate.

import torch
from mjlab.scene import Scene
from mjlab.entity import Entity
from mjlab.sim.sim import Simulation, SimulationCfg
from mjlab.viewer.offscreen_renderer import OffscreenRenderer
from mjlab.viewer.viewer_config import ViewerConfig
from moviepy import ImageSequenceClip

# Load the processed motion
ref = np.load(str(MOTION_NPZ))
ref_fps = int(ref["fps"][0])
ref_body_pos = torch.from_numpy(ref["body_pos_w"]).to(DEVICE)      # (T, B, 3)
ref_body_quat = torch.from_numpy(ref["body_quat_w"]).to(DEVICE)    # (T, B, 4)
ref_joint_pos = torch.from_numpy(ref["joint_pos"]).to(DEVICE)      # (T, J)
ref_joint_vel = torch.from_numpy(ref["joint_vel"]).to(DEVICE)      # (T, J)
ref_body_lin_vel = torch.from_numpy(ref["body_lin_vel_w"]).to(DEVICE)
ref_body_ang_vel = torch.from_numpy(ref["body_ang_vel_w"]).to(DEVICE)

# Build a minimal simulation + scene for rendering
sim_cfg = SimulationCfg()
sim_cfg.mujoco.timestep = 1.0 / ref_fps

scene = Scene(unitree_g1_flat_tracking_env_cfg().scene, device=DEVICE)
model = scene.compile()
sim = Simulation(num_envs=1, cfg=sim_cfg, model=model, device=DEVICE)
scene.initialize(sim.mj_model, sim.model, sim.data)

viewer_cfg = ViewerConfig(
    height=480, width=640,
    origin_type=ViewerConfig.OriginType.ASSET_ROOT,
    distance=2.0, elevation=-5.0, azimuth=20,
)
renderer = OffscreenRenderer(model=sim.mj_model, cfg=viewer_cfg, scene=scene)
renderer.initialize()

robot: Entity = scene["robot"]
scene.reset()

# Replay each frame
ref_frames = []
for t in range(ref_body_pos.shape[0]):
    # Set root state (body index 0 = root)
    root_states = robot.data.default_root_state.clone()
    root_states[:, 0:3] = ref_body_pos[t, 0:1]
    root_states[:, :2] += scene.env_origins[:, :2]
    root_states[:, 3:7] = ref_body_quat[t, 0:1]
    root_states[:, 7:10] = ref_body_lin_vel[t, 0:1]
    root_states[:, 10:] = ref_body_ang_vel[t, 0:1]
    robot.write_root_state_to_sim(root_states)

    # Set joint state
    jpos = robot.data.default_joint_pos.clone()
    jvel = robot.data.default_joint_vel.clone()
    jpos[0, :ref_joint_pos.shape[1]] = ref_joint_pos[t]
    jvel[0, :ref_joint_vel.shape[1]] = ref_joint_vel[t]
    robot.write_joint_state_to_sim(jpos, jvel)

    sim.forward()
    scene.update(sim.mj_model.opt.timestep)
    renderer.update(sim.data)
    ref_frames.append(renderer.render())

# Create and display video
ref_video = "/tmp/reference_motion.mp4"
ref_clip = ImageSequenceClip(ref_frames, fps=ref_fps)
ref_clip.write_videofile(ref_video, preset="ultrafast", logger=None)
print(f"Reference motion: {len(ref_frames)} frames, {len(ref_frames)/ref_fps:.2f}s @ {ref_fps} fps")
show_mp4(ref_video)

---
## Part 1: Understanding the Motion Imitation Pipeline

Before training, study the key components that make sim-to-sim (and eventually sim-to-real) transfer possible.

### 1.0 Reference Motion

A **reference motion** is a pre-recorded kinematic trajectory that specifies the desired pose of the humanoid at every timestep. Concretely, for each timestep $t$ in the clip, the reference motion provides the following quantities (all expressed in the Z-up world frame):

- **Root (anchor body) state:**
  - Position $\mathbf{p}^{\mathrm{ref}}_t \in \mathbb{R}^3$ — the global 3D position of the robot's torso (anchor body).
  - Orientation $\mathbf{q}^{\mathrm{ref}}_t \in SO(3)$ — the global orientation of the torso, represented as a quaternion or rotation matrix.
- **Body (link) states** for each of the 14 tracked bodies $b \in \{1,\dots,14\}$:
  - Position $\mathbf{x}^{\mathrm{ref}}_{b,t} \in \mathbb{R}^3$ — the 3D position of body $b$ relative to the root.
  - Orientation $\mathbf{R}^{\mathrm{ref}}_{b,t} \in SO(3)$ — the orientation of body $b$.
  - Linear velocity $\mathbf{v}^{\mathrm{ref}}_{b,t} \in \mathbb{R}^3$ and angular velocity $\boldsymbol{\omega}^{\mathrm{ref}}_{b,t} \in \mathbb{R}^3$.
- **Joint positions** $\boldsymbol{\theta}^{\mathrm{ref}}_t$ and **joint velocities** $\dot{\boldsymbol{\theta}}^{\mathrm{ref}}_t$ — the reference joint-space configuration, which can be used as feedforward targets for the actor.

The reference motion serves two purposes: (1) it defines the **reward signal** — the policy is rewarded for matching these reference quantities as closely as possible (Section 1.1), and (2) it provides **motion goal conditions** to the actor — the policy observes the current reference joint positions/velocities and the reference anchor orientation expressed in the body frame as a goal input, so it knows *what motion to execute next* (Section 1.2).

During training, the environment steps through the reference clip in lockstep with the simulation. At each timestep, the reference quantities at the current time index are compared against the robot's actual state to compute tracking rewards.

### 1.1 Reward Function

The reward encourages the policy to track the reference motion as closely as possible. Each **tracking reward term** uses an **exponential kernel** on the squared error between the robot's actual state and the corresponding reference quantity:

$$r = \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{x}^{\mathrm{ref}}\|^2}{\sigma^2}\right)$$

where $\mathbf{x}$ is the robot's actual quantity, $\mathbf{x}^{\mathrm{ref}}$ is the corresponding reference value, and $\sigma$ controls sensitivity. Perfect tracking ($\mathbf{x} = \mathbf{x}^{\mathrm{ref}}$) yields $r = 1$; large errors give $r \approx 0$.

The total reward is a weighted sum of tracking terms and smoothness/safety penalties. We separate them below.

**Tracking reward terms** are computed based on deviations of robot motion trajectory from the reference motion:

| Term | Weight | $\sigma$ | Error |
|------|--------|----------|-------|
| Root position | 0.5 | 0.3 m | $\|\mathbf{p}_t - \mathbf{p}^{\mathrm{ref}}_t\|^2$: global torso position vs. reference |
| Root orientation | 0.5 | 0.4 rad | $\|\mathbf{q}_t \ominus \mathbf{q}^{\mathrm{ref}}_t\|^2$: global torso orientation vs. reference |
| Body positions | 1.0 | 0.3 m | $\frac{1}{14}\sum_{b} \|\mathbf{x}_{b,t} - \mathbf{x}^{\mathrm{ref}}_{b,t}\|^2$: 14 tracked body positions (relative to root) vs. reference |
| Body orientations | 1.0 | 0.4 rad | $\frac{1}{14}\sum_{b} \|\mathbf{R}_{b,t} \ominus \mathbf{R}^{\mathrm{ref}}_{b,t}\|^2$: 14 tracked body orientations vs. reference |
| Body linear vel. | 1.0 | 1.0 m/s | $\frac{1}{14}\sum_{b} \|\mathbf{v}_{b,t} - \mathbf{v}^{\mathrm{ref}}_{b,t}\|^2$: body linear velocities vs. reference |
| Body angular vel. | 1.0 | 3.14 rad/s | $\frac{1}{14}\sum_{b} \|\boldsymbol{\omega}_{b,t} - \boldsymbol{\omega}^{\mathrm{ref}}_{b,t}\|^2$: body angular velocities vs. reference |

**Smoothness and safety penalties** encourage motion smoothness and non-interpenetration:

| Term | Weight | Description |
|------|--------|-------------|
| Action rate | -0.1 | $\|\mathbf{a}_t - \mathbf{a}_{t-1}\|^2$: penalizes large changes in consecutive actions to encourage smooth control signals |
| Joint limits | -10.0 | Penalty when joints approach their mechanical limits |
| Self-collision | -10.0 | Penalty for contact between the robot's own body parts (see code) |

### 1.2 Observation Space & Deployable Observations

A key challenge in sim-to-real transfer is that **not all simulator states are measurable on real hardware**. To address this, the framework uses an **asymmetric actor-critic** architecture where the actor and critic receive different observation vectors.

The **actor** (deployed policy) only sees observations available from onboard sensors. **Observation noise** is injected during training to simulator readings in order to simulate noisy observations from real-world sensors. Observation noise consists of small random perturbations (drawn uniformly from the ranges shown below) added to the actor's observations at every timestep. This forces the policy to learn control strategies that are robust to the imprecise, noisy measurements it will encounter on real hardware.

| Observation | Source on Real Robot | Description | Noise ($\pm$) |
|-------------|---------------------|-------------|---------------|
| Reference joint pos/vel | Motion planner | Current-frame reference $\boldsymbol{\theta}^{\mathrm{ref}}_t, \dot{\boldsymbol{\theta}}^{\mathrm{ref}}_t$ | — |
| Ref. anchor ori. (body frame) | Motion planner + IMU | Reference torso orientation in the robot's body frame (first two columns of the rotation matrix) | 0.05 (per entry) |
| Base angular velocity | IMU gyroscope | Angular velocity of the torso | 0.2 rad/s |
| Joint positions (rel. to default) | Joint encoders | $\boldsymbol{\theta}_t - \boldsymbol{\theta}_{\mathrm{default}}$ | 0.01 rad |
| Joint velocities | Encoder differentiation | $\dot{\boldsymbol{\theta}}_t$ | 0.5 rad/s |
| Previous actions | Known (own output) | $\mathbf{a}_{t-1}$ | — |

The **critic** (used only during training) additionally observes **privileged** information that is not available on real hardware: base linear velocity, the reference anchor position expressed in the robot's anchor frame (i.e., the root position error), and full body positions and orientations. This privileged information gives the critic a more accurate picture of the true robot state, leading to lower-variance value estimates during PPO training, while the actor learns to operate without it.

### 1.3 Terminal Conditions

Early termination is critical for efficient learning. When the robot enters an irrecoverable state (e.g., falling), continuing the episode wastes samples. The environment uses these termination criteria:

| Condition | Threshold | Description |
|-----------|-----------|-------------|
| `anchor_pos` ($z$-only) | 0.25 m | Root height ($z$-coordinate) deviates too far from reference $|p_{z,t} - p^{\mathrm{ref}}_{z,t}| > 0.25$ m |
| `anchor_ori` | 0.8 | Root tilt deviates too far from reference, measured as the difference in the $z$-component of gravity projected into the robot vs. reference anchor frames (unitless) |
| `ee_body_pos` ($z$-only) | 0.25 m | End-effector height ($z$-coordinate) deviates too far from reference |
| `time_out` | Episode end | Natural end of episode (not a failure) |

Note that the position-based terminations use **$z$-only** (vertical/height) thresholds rather than full 3D Euclidean distance. This is because vertical deviations indicate catastrophic failures (falling, collapsing), while horizontal ($xy$) drift is a softer tracking error that the reward can gradually correct.

### 1.4 Domain Randomization

Domain randomization injects variation during training so the policy must be robust to a range of physical conditions. This is essential for sim-to-sim (and sim-to-real) transfer because the deployment environment will inevitably have different physical parameters than the training simulator.

| Parameter | Range | When Applied |
|-----------|-------|---------------|
| Foot friction | 0.3–1.2 (absolute coefficient) | Startup (once per environment) |
| Base COM offset | ±2.5 cm ($x$), ±5 cm ($y$,$z$) | Startup (once per environment) |
| Joint encoder bias | ±0.01 rad | Startup (once per environment) |
| External push | ±0.5 m/s ($xy$), ±0.2 m/s ($z$); ±0.52 rad/s (roll, pitch), ±0.78 rad/s (yaw) | Every 1–3 s |
| Initial pose perturbation | ±5 cm ($xy$), ±1 cm ($z$); ±0.1 rad (roll, pitch), ±0.2 rad (yaw) | Episode start |
| Initial velocity perturbation | Same ranges as external push | Episode start |
| Initial joint perturbation | ±0.1 rad | Episode start |


---
## Part 2: Training

### 2.0 Implementation

Before training, you need to implement several missing components in `src/tasks/tracking/mdp/`. Open these files and fill in the `TODO` stubs.

**Q1: Reward Functions** (`src/tasks/tracking/mdp/rewards.py`)

1. **`motion_global_anchor_orientation_error_exp`**: Root orientation tracking reward. Use `quat_error_magnitude` to compute the error between `command.anchor_quat_w` (reference) and `command.robot_anchor_quat_w` (robot). Apply the exponential kernel $\exp(-e^2/\sigma^2)$.

2. **`motion_relative_body_orientation_error_exp`**: Body orientation tracking reward. Use `quat_error_magnitude` on `command.body_quat_relative_w` and `command.robot_body_quat_w`, indexed by `body_indexes`. Take the mean across bodies before applying the exponential kernel.

**Q2: Observation Functions** (`src/tasks/tracking/mdp/observations.py`)

1. **`motion_anchor_ori_b`**: Compute the reference anchor orientation in the robot's body frame. Use `subtract_frame_transforms` to get the relative orientation, convert to a rotation matrix with `matrix_from_quat`, extract the first two columns, and flatten to shape `(num_envs, 6)`.

2. **`robot_body_pos_b`**: Compute the positions of all tracked bodies in the robot's anchor frame. Expand the anchor pose to match the number of bodies, use `subtract_frame_transforms`, and flatten to shape `(num_envs, num_bodies * 3)`.

**Q3: Terminal Conditions** (`src/tasks/tracking/mdp/terminations.py`)

1. **`bad_anchor_pos_z_only`**: Terminate when the root height ($z$-coordinate) deviates too far from the reference. Compare only the $z$-component of `anchor_pos_w` and `robot_anchor_pos_w`. Return `True` where $|z_{\text{ref}} - z_{\text{robot}}| > \text{threshold}$.

2. **`bad_anchor_ori`**: Terminate based on orientation deviation using projected gravity. Project the gravity vector into both the reference and robot anchor frames using `quat_apply_inverse`, then compare the $z$-components. Return `True` where the absolute difference exceeds the threshold.

> **Hint:** Look at the already-implemented functions in each file for reference on how to use the `command` object and utility functions.

In [ ]:
#@title 2.1 Configure the Environment { display-mode: "form" }

NUM_ENVS = 4096  # GPU-parallel environments

env_cfg = unitree_g1_flat_tracking_env_cfg()

# Point to our short motion clip
motion_cmd = env_cfg.commands["motion"]
assert isinstance(motion_cmd, MotionCommandCfg), type(motion_cmd)
motion_cmd.motion_file = str(MOTION_NPZ.resolve())

env_cfg.scene.num_envs = NUM_ENVS
env_cfg.seed = SEED
env_cfg.episode_length_s = 2.0

print(f"--- Environment Configuration ---")
print(f"  Num envs:        {env_cfg.scene.num_envs}")
print(f"  Episode length:  {env_cfg.episode_length_s}s")
print(f"  Physics dt:      {env_cfg.sim.mujoco.timestep}s ({1/env_cfg.sim.mujoco.timestep:.0f} Hz)")
print(f"  Decimation:      {env_cfg.decimation}")
print(f"  Control dt:      {env_cfg.sim.mujoco.timestep * env_cfg.decimation}s ({1/(env_cfg.sim.mujoco.timestep * env_cfg.decimation):.0f} Hz)")
print(f"  Motion clip:     {MOTION_NPZ} ({duration:.2f}s)")
print(f"  Rewards:         {list(env_cfg.rewards.keys())}")
print(f"  Terminations:    {list(env_cfg.terminations.keys())}")
print(f"  Events (DR):     {list(env_cfg.events.keys())}")

In [ ]:
#@title 2.2 Configure PPO { display-mode: "form" }

MAX_ITERATIONS = 1500  # Enough for a short clip on T4

agent_cfg = unitree_g1_tracking_ppo_runner_cfg()
agent_cfg.algorithm.num_mini_batches = 8
agent_cfg.max_iterations = MAX_ITERATIONS
agent_cfg.save_interval = 500
agent_cfg.seed = SEED
agent_cfg.logger = "tensorboard"

samples_per_iter = NUM_ENVS * agent_cfg.num_steps_per_env
mini_batch_size = samples_per_iter // agent_cfg.algorithm.num_mini_batches

print(f"--- PPO Configuration ---")
print(f"  Max iterations:    {agent_cfg.max_iterations}")
print(f"  Steps per env:     {agent_cfg.num_steps_per_env}")
print(f"  Samples/iter:      {samples_per_iter:,}")
print(f"  Mini-batch size:   {mini_batch_size:,}")
print(f"  Learning epochs:   {agent_cfg.algorithm.num_learning_epochs}")
print(f"  Learning rate:     {agent_cfg.algorithm.learning_rate}")
print(f"  LR schedule:       {agent_cfg.algorithm.schedule}")
print(f"  Gamma:             {agent_cfg.algorithm.gamma}")
print(f"  Clip param:        {agent_cfg.algorithm.clip_param}")
print(f"  Entropy coef:      {agent_cfg.algorithm.entropy_coef}")

In [ ]:
#@title 2.3 Train { display-mode: "form" }

log_root = Path("logs/rsl_rl") / agent_cfg.experiment_name
log_dir = log_root / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_dir.mkdir(parents=True, exist_ok=True)
print(f"Log directory: {log_dir}")

# Create the training environment
env = ManagerBasedRlEnv(cfg=env_cfg, device=DEVICE)
env_wrapped = RslRlVecEnvWrapper(env, clip_actions=agent_cfg.clip_actions)

print(f"Observation space: {env.observation_space}")
print(f"Action space:      {env.action_space}")

# Create the runner and train
runner = MjlabOnPolicyRunner(
    env_wrapped,
    asdict(agent_cfg),
    str(log_dir),
    device=DEVICE,
)

print(f"\nStarting training for {MAX_ITERATIONS} iterations...")
runner.learn(
    num_learning_iterations=agent_cfg.max_iterations,
    init_at_random_ep_len=True,
)

print(f"\nTraining complete! Checkpoints saved in: {log_dir}")
env.close()

In [ ]:
#@title 2.4 Training Curves { display-mode: "form" }

from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def load_tb_scalars(log_dir: str) -> dict[str, list[tuple[int, float]]]:
    ea = EventAccumulator(log_dir)
    ea.Reload()
    scalars = {}
    for tag in ea.Tags().get("scalars", []):
        events = ea.Scalars(tag)
        scalars[tag] = [(e.step, e.value) for e in events]
    return scalars

scalars = load_tb_scalars(str(log_dir))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Mean reward
if "Train/mean_reward" in scalars:
    steps, vals = zip(*scalars["Train/mean_reward"])
    axes[0].plot(steps, vals, color="tab:blue")
    axes[0].set_title("Mean Episode Reward")
    axes[0].set_xlabel("Iteration")
    axes[0].grid(True, alpha=0.3)

# Episode length
if "Train/mean_episode_length" in scalars:
    steps, vals = zip(*scalars["Train/mean_episode_length"])
    axes[1].plot(steps, vals, color="tab:orange")
    axes[1].set_title("Mean Episode Length (steps)")
    axes[1].set_xlabel("Iteration")
    axes[1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

---
## Part 3: Sim-to-Sim Deployment

Now we deploy the trained policy with evaluation (play-mode) settings: no observation noise, no external pushes, and no initial root perturbation. The startup randomization of friction, COM and encoder bias stays on, so each evaluation environment has slightly different physics.

Key differences between training and deployment:

| Aspect | Training | Deployment |
|--------|----------|------------|
| Observation noise | Yes (uniform) | No |
| External pushes | Yes (every 1–3 s) | No |
| Friction / COM / encoder-bias randomization | Yes (once per environment) | Yes, same ranges (3.4 and 3.3.2); nominal in the 3.3.1 video |
| Episode start | Random time in motion (adaptive sampling) | From beginning |
| Initial root pose/velocity perturbation | Yes | No |
| Initial joint perturbation | ±0.1 rad | ±0.1 rad (3.4 and 3.3.2); none in the 3.3.1 video |
| Episode length | 2 s (set in 2.1) | One pass through the clip |
| Simulator | mujoco-warp (GPU) | mujoco-warp (3.2, 3.4) and vanilla mujoco (3.3) |

In [ ]:
#@title 3.1 Load Best Checkpoint { display-mode: "form" }

checkpoints = sorted(glob.glob(str(log_dir / "model_*.pt")), key=lambda f: int(Path(f).stem.split("_")[1]))
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found in {log_dir}")
best_ckpt = checkpoints[-1]
print(f"Loading checkpoint: {best_ckpt}")

In [ ]:
#@title 3.2 Deploy in Clean Simulator { display-mode: "form" }

# Create play-mode environment: no noise, no DR, start from beginning
play_env_cfg = unitree_g1_flat_tracking_env_cfg(play=True)
play_motion_cmd = play_env_cfg.commands["motion"]
assert isinstance(play_motion_cmd, MotionCommandCfg)
play_motion_cmd.motion_file = str(MOTION_NPZ.resolve())

play_env_cfg.scene.num_envs = 1

play_env = ManagerBasedRlEnv(cfg=play_env_cfg, device=DEVICE, render_mode="rgb_array")
play_env_wrapped = RslRlVecEnvWrapper(play_env, clip_actions=agent_cfg.clip_actions)

# Load the trained policy
eval_runner = MjlabOnPolicyRunner(
    play_env_wrapped,
    asdict(agent_cfg),
    device=DEVICE,
)
eval_runner.load(best_ckpt, map_location=DEVICE)
policy = eval_runner.get_inference_policy(device=DEVICE)

# Run inference and collect video frames
print("Running sim-to-sim deployment...")
obs = play_env_wrapped.get_observations()
frames = []
n_eval_steps = n_frames + 20  # full clip + buffer

for step in range(n_eval_steps):
    with torch.inference_mode():
        actions = policy(obs)
    obs, _, dones, _ = play_env_wrapped.step(actions)
    frame = play_env.render()
    if frame is not None:
        frames.append(frame)

play_env.close()

# Create video
def show_mp4(path: str, width: int = 640):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video width="{width}" controls autoplay loop>'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4">'
        f'</video>'
    ))

eval_video = "/tmp/sim2sim_deploy.mp4"
clip = ImageSequenceClip(frames, fps=50)
clip.write_videofile(eval_video, preset="ultrafast", logger=None)
print(f"Video saved: {eval_video} ({len(frames)} frames, {len(frames)/50:.1f}s)")
show_mp4(eval_video)

### 3.3 Deploy in Vanilla MuJoCo (Sim-to-Sim Transfer)

The previous cell used the training framework's `ManagerBasedRlEnv` (backed by `mujoco-warp`) for deployment.
Here we perform a true **sim-to-sim transfer**: the policy runs in **vanilla MuJoCo** (`mujoco` Python package, CPU)
with observations reconstructed manually — no training framework involved.

This mirrors real-world deployment where:
- The physics simulator is **different** from the training one
- Observations must be constructed from **raw sensor readings**
- There is **no** observation noise and no external pushes

Cell 3.3.1 runs one episode with nominal physics and renders a video. Cell 3.3.2 runs 16 episodes with the same randomization and termination conditions as the Warp evaluation in 3.4, and records the tracking errors.

In [ ]:
#@title 3.3.1 Deploy in Vanilla MuJoCo { display-mode: "form" }

import re
import mujoco
import numpy as np

# ── 1. Build the MuJoCo model via the Scene, using the ENV's sim config ──
#    This ensures solver/timestep/iterations match the training environment.
from mjlab.scene import Scene
from mjlab.sim.sim import Simulation

_scene = Scene(unitree_g1_flat_tracking_env_cfg().scene, device=DEVICE)
_compiled = _scene.compile()
_sim = Simulation(num_envs=1, cfg=env_cfg.sim, model=_compiled, device=DEVICE)
mj_model: mujoco.MjModel = _sim.mj_model

# Verify physics settings match the training config
DECIMATION = env_cfg.decimation   # 4
assert abs(mj_model.opt.timestep - env_cfg.sim.mujoco.timestep) < 1e-9, \
    f"timestep mismatch: {mj_model.opt.timestep} vs {env_cfg.sim.mujoco.timestep}"
print(f"Physics: dt={mj_model.opt.timestep}, decimation={DECIMATION}, "
      f"control_dt={mj_model.opt.timestep*DECIMATION}")
print(f"Solver: type={mj_model.opt.solver}, iter={mj_model.opt.iterations}, "
      f"ls_iter={mj_model.opt.ls_iterations}, integrator={mj_model.opt.integrator}")

mj_data = mujoco.MjData(mj_model)

# ── 2. Joint / actuator bookkeeping ──
NUM_JOINTS = 29
QPOS_OFFSET = 7   # free-joint uses qpos[0:7]
QVEL_OFFSET = 6   # free-joint uses qvel[0:6]

# Default joint positions (KNEES_BENT_KEYFRAME)
DEFAULT_JOINT_POS = np.array([
    -0.312, 0.0, 0.0, 0.669, -0.363, 0.0,   # left leg
    -0.312, 0.0, 0.0, 0.669, -0.363, 0.0,   # right leg
     0.0, 0.0, 0.0,                           # waist
     0.2, 0.2, 0.0, 0.6, 0.0, 0.0, 0.0,     # left arm
     0.2, -0.2, 0.0, 0.6, 0.0, 0.0, 0.0,    # right arm
], dtype=np.float64)

# Action scale per joint (from G1_ACTION_SCALE resolved to joint order)
ACTION_SCALE = np.array([
    0.547546, 0.350661, 0.547546, 0.350661, 0.438578, 0.438578,
    0.547546, 0.350661, 0.547546, 0.350661, 0.438578, 0.438578,
    0.547546, 0.438578, 0.438578,
    0.438578, 0.438578, 0.438578, 0.438578, 0.438578, 0.074501, 0.074501,
    0.438578, 0.438578, 0.438578, 0.438578, 0.438578, 0.074501, 0.074501,
], dtype=np.float64)

# Mapping: actuator index -> policy joint index
act_to_jnt = np.array([
    mj_model.jnt_qposadr[mj_model.actuator_trnid[i, 0]] - QPOS_OFFSET
    for i in range(mj_model.nu)
])

# Joint limits (joint 0 is the free joint), used to clip the initial joint perturbation
JOINT_LOW = mj_model.jnt_range[1:1+NUM_JOINTS, 0]
JOINT_HIGH = mj_model.jnt_range[1:1+NUM_JOINTS, 1]

# Torso link body id (anchor body) in MuJoCo and in the NPZ
TORSO_BODY_ID = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, "robot/torso_link")
TORSO_NPZ_IDX = TORSO_BODY_ID - 2  # NPZ stores robot bodies starting at MuJoCo body 2

# IMU sensor addresses
imu_ang_vel_adr = mj_model.sensor_adr[
    mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_SENSOR, "robot/imu_ang_vel")
]

# Body indices in the NPZ that correspond to the 14 tracked bodies
_eval_cfg = unitree_g1_flat_tracking_env_cfg(play=True)
TRACKED_BODY_NAMES = list(_eval_cfg.commands["motion"].body_names)
TRACKED_MJ_IDS = [mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, f"robot/{n}") for n in TRACKED_BODY_NAMES]
TRACKED_NPZ_IDS = [bid - 2 for bid in TRACKED_MJ_IDS]  # NPZ offset

# End-effector indices within the tracked body list
EE_NAMES_VANILLA = list(_eval_cfg.terminations["ee_body_pos"].params["body_names"])
EE_TRACKED_IDX = [TRACKED_BODY_NAMES.index(n) for n in EE_NAMES_VANILLA]

# Termination thresholds (same as the training / Warp evaluation environment)
ANCHOR_Z_THRESHOLD = _eval_cfg.terminations["anchor_pos"].params["threshold"]
ANCHOR_ORI_THRESHOLD = _eval_cfg.terminations["anchor_ori"].params["threshold"]
EE_Z_THRESHOLD = _eval_cfg.terminations["ee_body_pos"].params["threshold"]

# Startup randomization ranges (play mode keeps these, so the Warp evaluation uses them too)
FRICTION_RANGE = _eval_cfg.events["foot_friction"].params["ranges"]
COM_RANGES = _eval_cfg.events["base_com"].params["ranges"]
ENCODER_BIAS_RANGE = _eval_cfg.events["encoder_bias"].params["bias_range"]
JOINT_POS_RANGE = _eval_cfg.commands["motion"].joint_position_range
FOOT_GEOM_IDS = [
    g for g in range(mj_model.ngeom)
    if re.match(r"^robot/(left|right)_foot[1-7]_collision$",
                mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_GEOM, g) or "")
]
assert len(FOOT_GEOM_IDS) == 14, FOOT_GEOM_IDS
DEFAULT_GEOM_FRICTION = mj_model.geom_friction.copy()
DEFAULT_BODY_IPOS = mj_model.body_ipos.copy()

# ── 3. Load the trained actor (pure NumPy, no framework) ──
ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
actor_sd = ckpt["actor_state_dict"]

# Observation normalizer
obs_mean = actor_sd["obs_normalizer._mean"].numpy().flatten()   # (154,)
obs_var  = actor_sd["obs_normalizer._var"].numpy().flatten()    # (154,)
obs_std  = np.sqrt(obs_var + 1e-8)

# MLP weights
W0 = actor_sd["mlp.0.weight"].numpy(); b0 = actor_sd["mlp.0.bias"].numpy()
W1 = actor_sd["mlp.2.weight"].numpy(); b1 = actor_sd["mlp.2.bias"].numpy()
W2 = actor_sd["mlp.4.weight"].numpy(); b2 = actor_sd["mlp.4.bias"].numpy()
W3 = actor_sd["mlp.6.weight"].numpy(); b3 = actor_sd["mlp.6.bias"].numpy()

def elu(x, alpha=1.0):
    return np.where(x > 0, x, alpha * (np.exp(x) - 1))

def forward_actor(obs):
    """Run the actor MLP: obs (154,) -> action (29,)."""
    x = (obs - obs_mean) / obs_std
    x = elu(W0 @ x + b0)
    x = elu(W1 @ x + b1)
    x = elu(W2 @ x + b2)
    x = W3 @ x + b3
    return x

# ── 4. Load the reference motion ──
ref = np.load(str(MOTION_NPZ))
ref_fps = int(ref["fps"][0])
ref_joint_pos  = ref["joint_pos"]       # (T, 29)
ref_joint_vel  = ref["joint_vel"]       # (T, 29)
ref_body_pos_w = ref["body_pos_w"]      # (T, 30, 3)
ref_body_quat_w = ref["body_quat_w"]   # (T, 30, 4) wxyz
ref_body_lin_vel_w = ref["body_lin_vel_w"]  # (T, 30, 3)
ref_body_ang_vel_w = ref["body_ang_vel_w"]  # (T, 30, 3)

# ── 5. Quaternion helpers (wxyz convention, matching MuJoCo) ──
def quat_conjugate(q):
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_multiply(q1, q2):
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2,
    ])

def quat_rotate_vec(q, v):
    """Rotate vector v by quaternion q (wxyz)."""
    qv = np.array([0.0, v[0], v[1], v[2]])
    return quat_multiply(quat_multiply(q, qv), quat_conjugate(q))[1:]

def quat_apply_inverse(q, v):
    """Rotate vector v by the inverse of quaternion q (wxyz)."""
    return quat_rotate_vec(quat_conjugate(q), v)

def yaw_quat(q):
    """Keep only the yaw component of quaternion q (wxyz)."""
    w, x, y, z = q
    yaw = np.arctan2(2 * (w*z + x*y), 1 - 2 * (y*y + z*z))
    return np.array([np.cos(yaw / 2), 0.0, 0.0, np.sin(yaw / 2)])

def quat_error(q1, q2):
    """Rotation angle between quaternions (same as quat_error_magnitude)."""
    dots = np.abs(np.sum(q1 * q2, axis=-1))
    return 2.0 * np.arccos(np.clip(dots, 0.0, 1.0))

def quat_to_mat(q):
    """Convert wxyz quaternion to 3x3 rotation matrix."""
    w, x, y, z = q
    return np.array([
        [1-2*(y*y+z*z), 2*(x*y-w*z),   2*(x*z+w*y)],
        [2*(x*y+w*z),   1-2*(x*x+z*z), 2*(y*z-w*x)],
        [2*(x*z-w*y),   2*(y*z+w*x),   1-2*(x*x+y*y)],
    ])

# ── 6. Observation builder ──
def build_obs(mj_data, frame_idx, last_action, encoder_bias):
    """Construct the 154-dim actor observation from raw MuJoCo state."""
    # (58) Reference motion command: [ref_joint_pos, ref_joint_vel]
    cmd = np.concatenate([ref_joint_pos[frame_idx], ref_joint_vel[frame_idx]])

    # Robot anchor (torso_link) and reference anchor orientations in world frame
    robot_quat = mj_data.xquat[TORSO_BODY_ID].copy()
    ref_quat = ref_body_quat_w[frame_idx, TORSO_NPZ_IDX].copy()

    # (6) Relative orientation as first 2 columns of rotation matrix
    anchor_quat_b = quat_multiply(quat_conjugate(robot_quat), ref_quat)
    anchor_ori_b = quat_to_mat(anchor_quat_b)[:, :2].flatten()

    # (3) IMU gyroscope (body-frame angular velocity)
    base_ang_vel = mj_data.sensordata[imu_ang_vel_adr:imu_ang_vel_adr+3].copy()

    # (29+29) Joint state relative to defaults. The encoders read position + bias,
    # as in mjlab's joint_pos_rel(biased=True).
    joint_pos_rel = mj_data.qpos[QPOS_OFFSET:QPOS_OFFSET+NUM_JOINTS] - DEFAULT_JOINT_POS + encoder_bias
    joint_vel_rel = mj_data.qvel[QVEL_OFFSET:QVEL_OFFSET+NUM_JOINTS].copy()

    # Concatenate in the exact order the actor expects (154 dims)
    return np.concatenate([
        cmd,            # (58) command
        anchor_ori_b,   # (6)  motion_anchor_ori_b
        base_ang_vel,   # (3)  base_ang_vel
        joint_pos_rel,  # (29) joint_pos
        joint_vel_rel,  # (29) joint_vel
        last_action,    # (29) actions
    ]).astype(np.float32)

# ── 7. Tracking errors and terminations (same definitions as the Warp environment) ──
def tracking_errors(mj_data, f):
    """Errors of the current robot state against reference frame f, and termination flag.

    Matches MotionCommand._update_metrics, mdp/metrics.py and mdp/terminations.py.
    """
    robot_anchor_pos = mj_data.xpos[TORSO_BODY_ID].copy()
    robot_anchor_quat = mj_data.xquat[TORSO_BODY_ID].copy()
    ref_anchor_pos = ref_body_pos_w[f, TORSO_NPZ_IDX]
    ref_anchor_quat = ref_body_quat_w[f, TORSO_NPZ_IDX]

    robot_body_pos = mj_data.xpos[TRACKED_MJ_IDS].copy()
    robot_body_quat = mj_data.xquat[TRACKED_MJ_IDS].copy()
    ref_body_pos = ref_body_pos_w[f, TRACKED_NPZ_IDS]
    ref_body_quat = ref_body_quat_w[f, TRACKED_NPZ_IDS]

    # body_pos_relative_w / body_quat_relative_w: the reference pose moved to the
    # robot's anchor xy position and heading (see MotionCommand._update_command)
    delta_pos = robot_anchor_pos.copy()
    delta_pos[2] = ref_anchor_pos[2]
    delta_ori = yaw_quat(quat_multiply(robot_anchor_quat, quat_conjugate(ref_anchor_quat)))
    body_pos_rel = delta_pos + np.array([quat_rotate_vec(delta_ori, p - ref_anchor_pos) for p in ref_body_pos])
    body_quat_rel = np.array([quat_multiply(delta_ori, q) for q in ref_body_quat])

    robot_rel = robot_body_pos - robot_anchor_pos
    ref_rel = ref_body_pos - ref_anchor_pos
    robot_jvel = mj_data.qvel[QVEL_OFFSET:QVEL_OFFSET+NUM_JOINTS]

    errors = {
        "anchor_pos": np.linalg.norm(ref_anchor_pos - robot_anchor_pos),
        "anchor_ori": quat_error(ref_anchor_quat, robot_anchor_quat),
        "body_pos": np.linalg.norm(body_pos_rel - robot_body_pos, axis=-1).mean(),
        "body_ori": quat_error(body_quat_rel, robot_body_quat).mean(),
        "mpkpe": np.linalg.norm(ref_body_pos - robot_body_pos, axis=-1).mean(),
        "r_mpkpe": np.linalg.norm(ref_rel - robot_rel, axis=-1).mean(),
        "jvel": np.linalg.norm(ref_joint_vel[f] - robot_jvel),
        "ee_pos": np.linalg.norm(body_pos_rel[EE_TRACKED_IDX] - robot_body_pos[EE_TRACKED_IDX], axis=-1).mean(),
        "ee_ori": quat_error(body_quat_rel[EE_TRACKED_IDX], robot_body_quat[EE_TRACKED_IDX]).mean(),
    }

    gravity = np.array([0.0, 0.0, -1.0])
    terminated = (
        abs(ref_anchor_pos[2] - robot_anchor_pos[2]) > ANCHOR_Z_THRESHOLD
        or abs(quat_apply_inverse(ref_anchor_quat, gravity)[2]
               - quat_apply_inverse(robot_anchor_quat, gravity)[2]) > ANCHOR_ORI_THRESHOLD
        or np.any(np.abs(body_pos_rel[EE_TRACKED_IDX, 2] - robot_body_pos[EE_TRACKED_IDX, 2]) > EE_Z_THRESHOLD)
    )
    return errors, terminated

# ── 8. One deployment episode ──
def run_vanilla_episode(n_steps, rng=None, renderer=None):
    """Run the policy for up to n_steps control steps in vanilla MuJoCo.

    rng=None uses nominal physics. Otherwise friction, torso COM, encoder bias and
    the initial joint positions are randomized with the same ranges as the Warp
    play-mode evaluation. The episode stops at the first termination.
    """
    mj_model.geom_friction[:] = DEFAULT_GEOM_FRICTION
    mj_model.body_ipos[:] = DEFAULT_BODY_IPOS
    encoder_bias = np.zeros(NUM_JOINTS)
    joint_noise = np.zeros(NUM_JOINTS)
    if rng is not None:
        mj_model.geom_friction[FOOT_GEOM_IDS, 0] = rng.uniform(*FRICTION_RANGE)  # shared by all foot geoms
        for axis, (lo, hi) in COM_RANGES.items():
            mj_model.body_ipos[TORSO_BODY_ID, axis] += rng.uniform(lo, hi)
        encoder_bias = rng.uniform(*ENCODER_BIAS_RANGE, size=NUM_JOINTS)
        joint_noise = rng.uniform(*JOINT_POS_RANGE, size=NUM_JOINTS)

    # Initialize from the first reference frame, as the Warp environment does on reset
    mujoco.mj_resetData(mj_model, mj_data)
    root_quat = ref_body_quat_w[0, 0]
    mj_data.qpos[0:3] = ref_body_pos_w[0, 0]         # pelvis (root body) position
    mj_data.qpos[3:7] = root_quat                    # pelvis quaternion (wxyz)
    mj_data.qvel[0:3] = ref_body_lin_vel_w[0, 0]     # world-frame linear velocity
    mj_data.qvel[3:6] = quat_apply_inverse(root_quat, ref_body_ang_vel_w[0, 0])  # body-frame angular velocity
    mj_data.qpos[QPOS_OFFSET:QPOS_OFFSET+NUM_JOINTS] = np.clip(ref_joint_pos[0] + joint_noise, JOINT_LOW, JOINT_HIGH)
    mj_data.qvel[QVEL_OFFSET:QVEL_OFFSET+NUM_JOINTS] = ref_joint_vel[0]
    mujoco.mj_forward(mj_model, mj_data)

    last_action = np.zeros(NUM_JOINTS, dtype=np.float32)
    errors = {}
    frames = []
    terminated = False
    for t in range(n_steps):
        obs = build_obs(mj_data, t, last_action, encoder_bias)
        action = forward_actor(obs)
        last_action = action.astype(np.float32)

        # Joint position target = action * scale + default_pos, corrected for encoder bias
        joint_targets = action * ACTION_SCALE + DEFAULT_JOINT_POS - encoder_bias
        for act_i in range(mj_model.nu):
            mj_data.ctrl[act_i] = joint_targets[act_to_jnt[act_i]]

        # Step physics (decimation x physics_dt = control_dt)
        for _ in range(DECIMATION):
            mujoco.mj_step(mj_model, mj_data)

        if renderer is not None:
            renderer.update_scene(mj_data)
            frames.append(renderer.render().copy())

        # The reference advances one frame per control step, so compare with frame t+1
        step_errors, terminated = tracking_errors(mj_data, t + 1)
        if terminated:
            break
        for k, v in step_errors.items():
            errors.setdefault(k, []).append(v)

    mj_model.geom_friction[:] = DEFAULT_GEOM_FRICTION
    mj_model.body_ipos[:] = DEFAULT_BODY_IPOS
    return {"errors": errors, "episode_length": t + 1, "survived": not terminated, "frames": frames}

# ── 9. Nominal deployment with video ──
n_deploy_steps = ref_joint_pos.shape[0] - 1   # stop at the last reference frame
print(f"Running vanilla MuJoCo deployment for {n_deploy_steps} control steps "
      f"({n_deploy_steps/ref_fps:.2f}s) ...")
renderer = mujoco.Renderer(mj_model, height=480, width=640)
nominal_run = run_vanilla_episode(n_deploy_steps, rng=None, renderer=renderer)
renderer.close()
print(f"Survived: {nominal_run['survived']}  |  episode length: {nominal_run['episode_length']} steps")

sim2sim_frames = nominal_run["frames"]
sim2sim_video = "/tmp/sim2sim_vanilla_mujoco.mp4"
sim2sim_clip = ImageSequenceClip(sim2sim_frames, fps=ref_fps)
sim2sim_clip.write_videofile(sim2sim_video, preset="ultrafast", logger=None)
print(f"Vanilla MuJoCo video: {len(sim2sim_frames)} frames, "
      f"{len(sim2sim_frames)/ref_fps:.2f}s @ {ref_fps} fps")
show_mp4(sim2sim_video)

In [ ]:
#@title 3.3.2 Evaluate in Vanilla MuJoCo (16 runs) { display-mode: "form" }

# Run 16 episodes with the same randomization and terminations as the Warp
# evaluation in 3.4. Per-step errors are averaged over the runs that are still
# alive, as in 3.4.
NUM_VANILLA_RUNS = 16
NUM_EVAL_STEPS = n_frames - 1   # stop at the last reference frame

vanilla_rng = np.random.default_rng(SEED)
vanilla_runs = [run_vanilla_episode(NUM_EVAL_STEPS, rng=vanilla_rng) for _ in range(NUM_VANILLA_RUNS)]

vanilla_survival_rate = np.mean([r["survived"] for r in vanilla_runs])
vanilla_mean_ep_len = np.mean([r["episode_length"] for r in vanilla_runs])

def vanilla_per_step_mean(key):
    """Mean over alive runs at each step, up to the last step with an alive run."""
    n = max(len(r["errors"].get(key, [])) for r in vanilla_runs)
    table = np.full((NUM_VANILLA_RUNS, n), np.nan)
    for i, r in enumerate(vanilla_runs):
        vals = r["errors"].get(key, [])
        table[i, :len(vals)] = vals
    return list(np.nanmean(table, axis=0)) if n > 0 else []

vanilla_anchor_pos_errors = vanilla_per_step_mean("anchor_pos")
vanilla_anchor_ori_errors = vanilla_per_step_mean("anchor_ori")
vanilla_body_pos_errors = vanilla_per_step_mean("body_pos")
vanilla_body_ori_errors = vanilla_per_step_mean("body_ori")
vanilla_mpkpe_list = vanilla_per_step_mean("mpkpe")
vanilla_r_mpkpe_list = vanilla_per_step_mean("r_mpkpe")
vanilla_jvel_list = vanilla_per_step_mean("jvel")
vanilla_ee_pos_list = vanilla_per_step_mean("ee_pos")
vanilla_ee_ori_list = vanilla_per_step_mean("ee_ori")

print(f"Vanilla MuJoCo: {NUM_VANILLA_RUNS} runs x {NUM_EVAL_STEPS} steps")
print(f"  Survival rate:        {vanilla_survival_rate*100:.1f}%")
print(f"  Mean episode length:  {vanilla_mean_ep_len:.1f} steps")

### 3.4 Quantitative Evaluation

Evaluate the trained policy across 16 parallel environments in the training simulator (MuJoCo-Warp) and compare its tracking errors with the 16 vanilla MuJoCo runs from 3.3.2.

In [ ]:
from typing import cast
from src.tasks.tracking.mdp.commands import MotionCommand
from mjlab.utils.lab_api.math import quat_error_magnitude

NUM_EVAL_ENVS = 16
NUM_EVAL_STEPS = n_frames - 1  # stop before the clip wraps around and the robot is reset to frame 0

eval_env_cfg = unitree_g1_flat_tracking_env_cfg(play=True)
eval_motion_cmd = eval_env_cfg.commands["motion"]
assert isinstance(eval_motion_cmd, MotionCommandCfg)
eval_motion_cmd.motion_file = str(MOTION_NPZ.resolve())
eval_env_cfg.scene.num_envs = NUM_EVAL_ENVS

eval_env = ManagerBasedRlEnv(cfg=eval_env_cfg, device=DEVICE)
eval_env_wrapped = RslRlVecEnvWrapper(eval_env, clip_actions=agent_cfg.clip_actions)

eval_runner2 = MjlabOnPolicyRunner(eval_env_wrapped, asdict(agent_cfg), device=DEVICE)
eval_runner2.load(best_ckpt, map_location=DEVICE)
policy2 = eval_runner2.get_inference_policy(device=DEVICE)

# Fix: After reset(), body_pos_relative_w is all zeros because _update_command()
# has not run yet.  The ee_body_pos termination compares this against actual robot
# body positions, causing all envs to terminate at step 0.  Force the initialization
# by calling _update_command() with a pre-decremented time_steps so it increments
# back to 0 and computes body_pos_relative_w for the initial frame.
_cmd = cast(MotionCommand, eval_env.command_manager.get_term("motion"))
_cmd.time_steps -= 1
_cmd._update_command()

# Collect tracking errors
anchor_pos_errors, anchor_ori_errors = [], []
body_pos_errors, body_ori_errors = [], []
all_rewards = []
episode_lengths = torch.zeros(NUM_EVAL_ENVS, device=DEVICE)
active = torch.ones(NUM_EVAL_ENVS, dtype=torch.bool, device=DEVICE)

obs = eval_env_wrapped.get_observations()
for step in range(NUM_EVAL_STEPS):
    with torch.inference_mode():
        actions = policy2(obs)
    obs, rewards, dones, extras = eval_env_wrapped.step(actions)

    episode_lengths[active] += 1
    active = active & (dones == 0)

    # Skip metric collection when no environments are still active
    if not active.any():
        break

    cmd = cast(MotionCommand, eval_env.command_manager.get_term("motion"))

    anchor_pos_errors.append(
        torch.norm(cmd.anchor_pos_w - cmd.robot_anchor_pos_w, dim=-1)[active].mean().item()
    )
    anchor_ori_errors.append(
        quat_error_magnitude(cmd.anchor_quat_w, cmd.robot_anchor_quat_w)[active].mean().item()
    )
    body_pos_errors.append(
        torch.norm(cmd.body_pos_relative_w - cmd.robot_body_pos_w, dim=-1).mean(-1)[active].mean().item()
    )
    body_ori_errors.append(
        quat_error_magnitude(cmd.body_quat_relative_w, cmd.robot_body_quat_w).mean(-1)[active].mean().item()
    )
    all_rewards.append(rewards[active].mean().item())

eval_env.close()

warp_survival_rate = active.float().mean().item()
warp_mean_ep_len = episode_lengths.float().mean().item()

def _mean(xs):
    return np.mean(xs) if len(xs) else float("nan")

control_dt = env_cfg.sim.mujoco.timestep * env_cfg.decimation
print("=" * 62)
print("  SIM-TO-SIM EVALUATION RESULTS")
print("=" * 62)
print(f"  {'':<30}{'Warp':>12}{'Vanilla MuJoCo':>18}")
print(f"  {'Runs':<30}{NUM_EVAL_ENVS:>12}{NUM_VANILLA_RUNS:>18}")
print(f"  {'Eval steps':<30}{NUM_EVAL_STEPS:>12}{NUM_EVAL_STEPS:>18}")
print(f"  {'Mean episode length (steps)':<30}{warp_mean_ep_len:>12.1f}{vanilla_mean_ep_len:>18.1f}")
print(f"  {'Survival rate (%)':<30}{warp_survival_rate*100:>12.1f}{vanilla_survival_rate*100:>18.1f}")
print(f"  --- Mean Tracking Errors ---")
print(f"  {'Anchor position (m)':<30}{_mean(anchor_pos_errors):>12.4f}{_mean(vanilla_anchor_pos_errors):>18.4f}")
print(f"  {'Anchor orientation (rad)':<30}{_mean(anchor_ori_errors):>12.4f}{_mean(vanilla_anchor_ori_errors):>18.4f}")
print(f"  {'Body position (m)':<30}{_mean(body_pos_errors):>12.4f}{_mean(vanilla_body_pos_errors):>18.4f}")
print(f"  {'Body orientation (rad)':<30}{_mean(body_ori_errors):>12.4f}{_mean(vanilla_body_ori_errors):>18.4f}")
print(f"  {'Mean step reward (Warp only)':<30}{_mean(all_rewards):>12.4f}")
print("=" * 62)

# Plot error over time (MuJoCo-Warp + Vanilla MuJoCo)
if anchor_pos_errors:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    time_axis = np.arange(len(anchor_pos_errors)) * control_dt
    vanilla_time_axis = np.arange(len(vanilla_anchor_pos_errors)) * control_dt

    axes[0].plot(time_axis, anchor_pos_errors, label="Anchor pos - Warp (m)")
    axes[0].plot(time_axis, body_pos_errors, label="Body pos - Warp (m)")
    axes[0].plot(vanilla_time_axis, vanilla_anchor_pos_errors, label="Anchor pos - Vanilla (m)", linestyle="--")
    axes[0].plot(vanilla_time_axis, vanilla_body_pos_errors, label="Body pos - Vanilla (m)", linestyle="--")
    axes[0].set_title("Position Tracking Error")
    axes[0].set_xlabel("Time (s)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time_axis, anchor_ori_errors, label="Anchor ori - Warp (rad)")
    axes[1].plot(time_axis, body_ori_errors, label="Body ori - Warp (rad)")
    axes[1].plot(vanilla_time_axis, vanilla_anchor_ori_errors, label="Anchor ori - Vanilla (rad)", linestyle="--")
    axes[1].plot(vanilla_time_axis, vanilla_body_ori_errors, label="Body ori - Vanilla (rad)", linestyle="--")
    axes[1].set_title("Orientation Tracking Error")
    axes[1].set_xlabel("Time (s)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No tracking data to plot (all envs terminated immediately).")

In [ ]:
#@title 3.5 Detailed Metrics (MPKPE, R-MPKPE, Joint Vel, End-Effector) { display-mode: "form" }

from src.tasks.tracking.mdp.metrics import (
    compute_mpkpe,
    compute_root_relative_mpkpe,
    compute_joint_velocity_error,
    compute_ee_position_error,
    compute_ee_orientation_error,
)

EE_BODY_NAMES = (
    "left_ankle_roll_link", "right_ankle_roll_link",
    "left_wrist_yaw_link", "right_wrist_yaw_link",
)

control_dt = env_cfg.sim.mujoco.timestep * env_cfg.decimation

# --- Run evaluation with metric collection ---
metric_env_cfg = unitree_g1_flat_tracking_env_cfg(play=True)
metric_motion_cmd = metric_env_cfg.commands["motion"]
assert isinstance(metric_motion_cmd, MotionCommandCfg)
metric_motion_cmd.motion_file = str(MOTION_NPZ.resolve())
metric_env_cfg.scene.num_envs = NUM_EVAL_ENVS

metric_env = ManagerBasedRlEnv(cfg=metric_env_cfg, device=DEVICE)
metric_env_wrapped = RslRlVecEnvWrapper(metric_env, clip_actions=agent_cfg.clip_actions)

metric_runner = MjlabOnPolicyRunner(metric_env_wrapped, asdict(agent_cfg), device=DEVICE)
metric_runner.load(best_ckpt, map_location=DEVICE)
metric_policy = metric_runner.get_inference_policy(device=DEVICE)

# Initialize command (same fix as before)
_cmd = cast(MotionCommand, metric_env.command_manager.get_term("motion"))
_cmd.time_steps -= 1
_cmd._update_command()

# Collect metrics per step
mpkpe_list, r_mpkpe_list, jvel_list = [], [], []
ee_pos_list, ee_ori_list = [], []
metric_active = torch.ones(NUM_EVAL_ENVS, dtype=torch.bool, device=DEVICE)

obs = metric_env_wrapped.get_observations()
for step in range(NUM_EVAL_STEPS):
    with torch.inference_mode():
        actions = metric_policy(obs)
    obs, _, dones, _ = metric_env_wrapped.step(actions)
    metric_active = metric_active & (dones == 0)
    if not metric_active.any():
        break

    cmd = cast(MotionCommand, metric_env.command_manager.get_term("motion"))
    mpkpe_list.append(compute_mpkpe(cmd)[metric_active].mean().item())
    r_mpkpe_list.append(compute_root_relative_mpkpe(cmd)[metric_active].mean().item())
    jvel_list.append(compute_joint_velocity_error(cmd)[metric_active].mean().item())
    ee_pos_list.append(compute_ee_position_error(cmd, EE_BODY_NAMES)[metric_active].mean().item())
    ee_ori_list.append(compute_ee_orientation_error(cmd, EE_BODY_NAMES)[metric_active].mean().item())

metric_env.close()

# --- Print summary ---
print("=" * 65)
print("  DETAILED EVALUATION METRICS")
print("=" * 65)
print(f"  {'Metric':<30} {'Warp':>10} {'Vanilla MuJoCo':>16}")
print("-" * 65)
if mpkpe_list:
    print(f"  {'MPKPE (world frame)':<30} {np.mean(mpkpe_list):>10.4f} m {np.mean(vanilla_mpkpe_list):>12.4f} m")
    print(f"  {'R-MPKPE (root-relative)':<30} {np.mean(r_mpkpe_list):>10.4f} m {np.mean(vanilla_r_mpkpe_list):>12.4f} m")
    print(f"  {'Joint velocity error':<30} {np.mean(jvel_list):>10.4f}   {np.mean(vanilla_jvel_list):>12.4f}  ")
    print(f"  {'EE position error':<30} {np.mean(ee_pos_list):>10.4f} m {np.mean(vanilla_ee_pos_list):>12.4f} m")
    print(f"  {'EE orientation error':<30} {np.mean(ee_ori_list):>10.4f}   {np.mean(vanilla_ee_ori_list):>12.4f}  ")
else:
    print("  (all environments terminated before metrics could be collected)")
print("=" * 65)

# --- Plot metrics over time ---
if mpkpe_list:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    time_axis = np.arange(len(mpkpe_list)) * control_dt
    vanilla_time_axis = np.arange(len(vanilla_mpkpe_list)) * control_dt

    axes[0].plot(time_axis, mpkpe_list, label="MPKPE - Warp")
    axes[0].plot(time_axis, r_mpkpe_list, label="R-MPKPE - Warp")
    axes[0].plot(vanilla_time_axis, vanilla_mpkpe_list, label="MPKPE - Vanilla", linestyle="--")
    axes[0].plot(vanilla_time_axis, vanilla_r_mpkpe_list, label="R-MPKPE - Vanilla", linestyle="--")
    axes[0].set_title("Keybody Position Error")
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Error (m)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time_axis, ee_pos_list, label="EE pos - Warp (m)")
    axes[1].plot(time_axis, ee_ori_list, label="EE ori - Warp (rad)")
    axes[1].plot(vanilla_time_axis, vanilla_ee_pos_list, label="EE pos - Vanilla (m)", linestyle="--")
    axes[1].plot(vanilla_time_axis, vanilla_ee_ori_list, label="EE ori - Vanilla (rad)", linestyle="--")
    axes[1].set_title("End-Effector Tracking Error")
    axes[1].set_xlabel("Time (s)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(time_axis, jvel_list, label="Joint vel - Warp", color="tab:red")
    axes[2].plot(vanilla_time_axis, vanilla_jvel_list, label="Joint vel - Vanilla", color="tab:red", linestyle="--")
    axes[2].set_title("Joint Velocity Error")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylabel("Error (rad/s)")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

---
## Part 4: Ablation Experiments

In this section you will modify the training configuration to study the impact of observation noise, domain randomization, and terminal conditions. For each ablation, re-train from scratch and compare sim-to-sim deployment performance.

The `train_and_evaluate()` helper function below trains a policy and evaluates it in MuJoCo-Warp play mode (16 environments, same settings as Section 3.4), plotting training curves automatically.

In [ ]:
def train_and_evaluate(
    env_cfg: ManagerBasedRlEnvCfg,
    agent_cfg: RslRlOnPolicyRunnerCfg,
    motion_npz: Path,
    n_frames: int,
    experiment_label: str,
    max_iterations: int = 1500,
    num_envs: int = 4096,
    device: str = DEVICE,
) -> dict:
    """Train a policy and evaluate it in play mode. Returns metrics dict."""
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

    # --- Train ---
    env_cfg.scene.num_envs = num_envs
    agent_cfg.max_iterations = max_iterations
    agent_cfg.logger = "tensorboard"

    log_dir = Path("logs/rsl_rl") / experiment_label / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    log_dir.mkdir(parents=True, exist_ok=True)

    train_env = ManagerBasedRlEnv(cfg=env_cfg, device=device)
    train_wrapped = RslRlVecEnvWrapper(train_env, clip_actions=agent_cfg.clip_actions)
    train_runner = MjlabOnPolicyRunner(train_wrapped, asdict(agent_cfg), str(log_dir), device=device)
    train_runner.learn(num_learning_iterations=max_iterations, init_at_random_ep_len=True)
    train_env.close()

    # --- Plot training curves ---
    ea = EventAccumulator(str(log_dir))
    ea.Reload()
    tb_scalars = {}
    for tag in ea.Tags().get("scalars", []):
        events = ea.Scalars(tag)
        tb_scalars[tag] = [(e.step, e.value) for e in events]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(f"Training Curves: {experiment_label}", fontsize=12)

    if "Train/mean_reward" in tb_scalars:
        steps, vals = zip(*tb_scalars["Train/mean_reward"])
        axes[0].plot(steps, vals, color="tab:blue")
        axes[0].set_title("Mean Episode Reward")
        axes[0].set_xlabel("Iteration")
        axes[0].grid(True, alpha=0.3)

    if "Train/mean_episode_length" in tb_scalars:
        steps, vals = zip(*tb_scalars["Train/mean_episode_length"])
        axes[1].plot(steps, vals, color="tab:orange")
        axes[1].set_title("Mean Episode Length (steps)")
        axes[1].set_xlabel("Iteration")
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Find best checkpoint
    ckpt = sorted(glob.glob(str(log_dir / "model_*.pt")), key=lambda f: int(Path(f).stem.split("_")[1]))[-1]

    # --- Evaluate in play mode ---
    play_cfg = unitree_g1_flat_tracking_env_cfg(play=True)
    play_cmd = play_cfg.commands["motion"]
    assert isinstance(play_cmd, MotionCommandCfg)
    play_cmd.motion_file = str(motion_npz.resolve())
    play_cfg.scene.num_envs = 16

    eval_env = ManagerBasedRlEnv(cfg=play_cfg, device=device)
    eval_wrapped = RslRlVecEnvWrapper(eval_env, clip_actions=agent_cfg.clip_actions)
    eval_runner = MjlabOnPolicyRunner(eval_wrapped, asdict(agent_cfg), device=device)
    eval_runner.load(ckpt, map_location=device)
    pol = eval_runner.get_inference_policy(device=device)

    # Fix: Initialize body_pos_relative_w which is left as zeros after reset().
    # Without this, the ee_body_pos termination fires spuriously at step 0.
    _cmd = cast(MotionCommand, eval_env.command_manager.get_term("motion"))
    _cmd.time_steps -= 1
    _cmd._update_command()

    obs = eval_wrapped.get_observations()
    ep_len = torch.zeros(16, device=device)
    active = torch.ones(16, dtype=torch.bool, device=device)
    body_errs = []

    for _ in range(n_frames - 1):  # stop before the clip wraps around
        with torch.inference_mode():
            actions = pol(obs)
        obs, rew, dones, _ = eval_wrapped.step(actions)

        ep_len[active] += 1
        active = active & (dones == 0)

        if not active.any():
            break

        cmd = cast(MotionCommand, eval_env.command_manager.get_term("motion"))
        body_errs.append(
            torch.norm(cmd.body_pos_relative_w - cmd.robot_body_pos_w, dim=-1)
            .mean(-1)[active].mean().item()
        )

    eval_env.close()

    return {
        "label": experiment_label,
        "mean_ep_len": ep_len.float().mean().item(),
        "survival_rate": active.float().mean().item(),
        "mean_body_pos_error": np.mean(body_errs) if body_errs else float("nan"),
        "body_pos_errors": body_errs,
        "log_dir": str(log_dir),
        "checkpoint": ckpt,
    }

### 4.1 Ablation: Observation Noise

Train a policy **without** observation noise and compare sim-to-sim transfer.

In [ ]:
# --- No observation noise ---
no_noise_env_cfg = unitree_g1_flat_tracking_env_cfg()
no_noise_cmd = no_noise_env_cfg.commands["motion"]
assert isinstance(no_noise_cmd, MotionCommandCfg)
no_noise_cmd.motion_file = str(MOTION_NPZ.resolve())
no_noise_env_cfg.seed = SEED
no_noise_env_cfg.episode_length_s = env_cfg.episode_length_s  # same as the baseline (Part 2.1)

# Disable observation noise on the actor
no_noise_env_cfg.observations["actor"].enable_corruption = False

no_noise_agent_cfg = unitree_g1_tracking_ppo_runner_cfg()
no_noise_agent_cfg.algorithm.num_mini_batches = 8
no_noise_agent_cfg.seed = SEED

results_no_noise = train_and_evaluate(
    no_noise_env_cfg, no_noise_agent_cfg, MOTION_NPZ, n_frames,
    experiment_label="ablation_no_obs_noise",
)
print(f"No obs noise  | Survival: {results_no_noise['survival_rate']*100:.0f}%  "
      f"| Body error: {results_no_noise['mean_body_pos_error']:.4f} m")

### 4.2 Ablation: Domain Randomization

Train a policy **without** domain randomization (no friction/COM/encoder randomization, no pushes, no initial-state perturbation).

In [ ]:
# --- No domain randomization ---
no_dr_env_cfg = unitree_g1_flat_tracking_env_cfg()
no_dr_cmd = no_dr_env_cfg.commands["motion"]
assert isinstance(no_dr_cmd, MotionCommandCfg)
no_dr_cmd.motion_file = str(MOTION_NPZ.resolve())
no_dr_env_cfg.seed = SEED
no_dr_env_cfg.episode_length_s = env_cfg.episode_length_s  # same as the baseline (Part 2.1)

# Remove all domain randomization events
no_dr_env_cfg.events.pop("push_robot", None)
no_dr_env_cfg.events.pop("base_com", None)
no_dr_env_cfg.events.pop("encoder_bias", None)
no_dr_env_cfg.events.pop("foot_friction", None)

# Also disable initial state randomization
no_dr_cmd.pose_range = {}
no_dr_cmd.velocity_range = {}
no_dr_cmd.joint_position_range = (0.0, 0.0)

no_dr_agent_cfg = unitree_g1_tracking_ppo_runner_cfg()
no_dr_agent_cfg.algorithm.num_mini_batches = 8
no_dr_agent_cfg.seed = SEED

results_no_dr = train_and_evaluate(
    no_dr_env_cfg, no_dr_agent_cfg, MOTION_NPZ, n_frames,
    experiment_label="ablation_no_domain_rand",
)
print(f"No DR         | Survival: {results_no_dr['survival_rate']*100:.0f}%  "
      f"| Body error: {results_no_dr['mean_body_pos_error']:.4f} m")

### 4.3 Ablation: Terminal Conditions

Train a policy **without** early termination (only time-out).

In [ ]:
# --- No early termination ---
no_term_env_cfg = unitree_g1_flat_tracking_env_cfg()
no_term_cmd = no_term_env_cfg.commands["motion"]
assert isinstance(no_term_cmd, MotionCommandCfg)
no_term_cmd.motion_file = str(MOTION_NPZ.resolve())
no_term_env_cfg.seed = SEED
no_term_env_cfg.episode_length_s = env_cfg.episode_length_s  # same as the baseline (Part 2.1)

# Keep only time_out, remove failure terminations
no_term_env_cfg.terminations = {
    k: v for k, v in no_term_env_cfg.terminations.items() if k == "time_out"
}

no_term_agent_cfg = unitree_g1_tracking_ppo_runner_cfg()
no_term_agent_cfg.algorithm.num_mini_batches = 8
no_term_agent_cfg.seed = SEED

results_no_term = train_and_evaluate(
    no_term_env_cfg, no_term_agent_cfg, MOTION_NPZ, n_frames,
    experiment_label="ablation_no_termination",
)
print(f"No term conds | Survival: {results_no_term['survival_rate']*100:.0f}%  "
      f"| Body error: {results_no_term['mean_body_pos_error']:.4f} m")

### 4.4 Compare All Ablations

In [ ]:
# Collect baseline results (from Part 3) into the same format as train_and_evaluate()
control_dt = env_cfg.sim.mujoco.timestep * env_cfg.decimation

results_baseline = {
    "label": "baseline (full pipeline)",
    "mean_ep_len": episode_lengths.float().mean().item(),
    "survival_rate": warp_survival_rate,
    "mean_body_pos_error": np.mean(body_pos_errors) if body_pos_errors else float("nan"),
    "body_pos_errors": body_pos_errors,
}

all_results = [results_baseline, results_no_noise, results_no_dr, results_no_term]

# --- Summary Table ---
print(f"{'Experiment':<30} {'Survival %':>12} {'Mean Ep Len':>12} {'Body Pos Err (m)':>18}")
print("-" * 74)
for r in all_results:
    print(f"{r['label']:<30} {r['survival_rate']*100:>11.1f}% {r['mean_ep_len']:>11.1f} {r['mean_body_pos_error']:>17.4f}")

# --- Error Trajectory Comparison ---
fig, ax = plt.subplots(figsize=(10, 5))
for r in all_results:
    if r["body_pos_errors"]:
        t = np.arange(len(r["body_pos_errors"])) * control_dt
        ax.plot(t, r["body_pos_errors"], label=r["label"], alpha=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Mean Body Position Error (m)")
ax.set_title("Sim-to-Sim Tracking Error: Ablation Comparison")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()